In [1]:
"""
JobSkillExtractor Module

This module defines the JobSkillExtractor class used to extract technical and soft skills
from unstructured job description text. It matches against predefined categories and allows
custom skill sets to support domain-specific needs.

Classes:
    JobSkillExtractor: Extracts skills from job descriptions, optionally grouped by category.

Typical Use Cases:
    - Parsing job descriptions for ATS/HR platforms
    - Matching job posts to candidate profiles
    - Highlighting required technical and soft skills
"""


import re
from typing import List, Dict, Optional, Union, Set


class JobSkillExtractor:
    """
    A class to extract relevant technical and soft skills from job descriptions.

    It matches skills across predefined categories such as Programming, AI/ML, Web Development,
    Cloud & DevOps, Databases, Visualization Tools, and Soft Skills. You can also pass a custom
    skill set.

    Attributes:
        skill_set (Dict[str, List[str]]): Dictionary of categorized skill keywords.
    """

    def __init__(self, skill_set: Optional[Dict[str, List[str]]] = None):
        """
        Initialize the extractor with a default or custom skill set.

        Args:
            skill_set (Optional[Dict[str, List[str]]]): Custom mapping of skill categories to skill terms.
        """
        self.skill_set = skill_set or {
            "Programming Languages": [
                "Python", "Java", "JavaScript", "TypeScript", "C++", "C#", "SQL", "Go", "Rust", "Kotlin", "Ruby"
            ],
            "AI / ML / NLP": [
                "Machine Learning", "Deep Learning", "TensorFlow", "PyTorch", "Scikit-learn", "Keras",
                "NLP", "Transformer", "OpenAI", "LangChain", "spaCy", "LLM", "BERT", "GPT"
            ],
            "Web Development": [
                "HTML", "CSS", "React", "Next.js", "Angular", "Vue.js", "Node.js", "Express", "Flask", "Django"
            ],
            "Cloud & DevOps": [
                "AWS", "Azure", "GCP", "Docker", "Kubernetes", "Terraform", "Jenkins", "GitHub Actions"
            ],
            "Databases": [
                "MySQL", "PostgreSQL", "MongoDB", "SQLite", "Redis", "Oracle", "Firestore", "ElasticSearch"
            ],
            "Visualization & BI": [
                "Power BI", "Tableau", "Plotly", "Seaborn", "Matplotlib", "Looker"
            ],
            "Soft Skills": [
                "Communication", "Problem Solving", "Teamwork", "Adaptability", "Time Management",
                "Critical Thinking", "Leadership", "Creativity"
            ]
        }

    def extract(self, text: str, return_grouped: bool = False) -> Union[List[str], Dict[str, List[str]]]:
        """
        Extracts skills mentioned in a job description.

        Args:
            text (str): Raw job description content.
            return_grouped (bool): If True, return a dictionary grouped by skill category.

        Returns:
            Union[List[str], Dict[str, List[str]]]:
                - List of matched skill names (default)
                - Dictionary of matched skills grouped by category (if return_grouped=True)
        """
        text_lower = text.lower()
        skills_found: Union[Set[str], Dict[str, Set[str]]] = {} if return_grouped else set()

        for category, skills in self.skill_set.items():
            for skill in skills:
                pattern = rf"\b{re.escape(skill.lower())}\b"
                if re.search(pattern, text_lower):
                    if return_grouped:
                        skills_found.setdefault(category, set()).add(skill)
                    else:
                        skills_found.add(skill)

        if return_grouped:
            return {cat: sorted(list(skills)) for cat, skills in skills_found.items() if skills}
        else:
            return sorted(list(skills_found)) if skills_found else []
        




['AWS', 'Communication', 'Django', 'Docker', 'Kubernetes', 'PostgreSQL', 'Python']
{'Programming Languages': ['Python'], 'Web Development': ['Django'], 'Cloud & DevOps': ['AWS', 'Docker', 'Kubernetes'], 'Databases': ['PostgreSQL'], 'Soft Skills': ['Communication']}


In [ ]:

if __name__ == "__main__":
    jd_text = """
    We're looking for a backend engineer skilled in Python, Django, and PostgreSQL.
    Experience with AWS, Docker, and Kubernetes is preferred.
    Strong communication and problem-solving abilities are a must.
    """

    extractor = JobSkillExtractor()
    print(extractor.extract(jd_text))  # Flat list
    print(extractor.extract(jd_text, return_grouped=True))  # Grouped by category

In [3]:
import pandas as pd

df = pd.read_csv("C:\Yousuf\DEPI\Technical\Mega Projects\Job_Recommendation_System\data\Processed\standardized_skills.csv")
df.to_json("..\data\skills.json", orient="records", force_ascii=False, indent=2)


In [1]:
import json
import re
from transformers import AutoTokenizer, AutoModelForTokenClassification
from typing import List, Dict, Optional, Union

class JobSkillExtractor:
    """
    Enhanced JobSkillExtractor using jjzha/jobbert_skill_extraction and keyword matching
    for extracting both technical and soft skills.
    
    Attributes:
        skill_set: Dictionary of categorized skills loaded from a JSON file (optional).
        skill_set_flat: Flattened list of skills for categorization (optional).
        tokenizer: Tokenizer for the pre-trained model.
        model: Pre-trained model for skill extraction.
        soft_skills: List of soft skills for keyword matching.
    """
    
    def __init__(self, skill_file: Optional[str] = None, model_name: str = "jjzha/jobbert_skill_extraction"):
        """
        Initialize with an optional skill set and the pre-trained model.
        
        Args:
            skill_file: Path to the JSON file containing skills (optional).
            model_name: Name of the Hugging Face model.
        """
        # Load skill set if provided
        self.skill_set = None
        self.skill_set_flat = []
        if skill_file:
            try:
                with open(skill_file, 'r') as f:
                    self.skill_set = json.load(f)
                self.skill_set_flat = [skill.lower() for skills in self.skill_set.values() for skill in skills]
            except FileNotFoundError:
                print(f"Warning: Skill file '{skill_file}' not found. Proceeding without skill set.")
        
        # Load the pre-trained model
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForTokenClassification.from_pretrained(model_name)
        
        # Define soft skills for keyword matching
        self.soft_skills = [
            "communication", "teamwork", "leadership", "problem-solving", "time management",
            "adaptability", "critical thinking", "creativity", "collaboration", "negotiation",
            "emotional intelligence", "conflict resolution", "public speaking", "writing",
            "decision making", "attention to detail", "strategic thinking", "empathy"
        ]
    
    def extract(self, text: str, return_grouped: bool = False, extract_potential: bool = False) -> Union[List[str], Dict[str, Union[List[str], Dict[str, List[str]]]]]:
        """
        Extract skills from a job description using the model and keyword matching.
        
        Args:
            text: Job description text.
            return_grouped: If True, return skills grouped by category (requires skill_file).
            extract_potential: If True, include all model-extracted skills as potential skills.
        
        Returns:
            Union[List[str], Dict[str, Union[List[str], Dict[str, List[str]]]]]:
                - If extract_potential is False: List or grouped dict of skills.
                - If extract_potential is True: Dict with known and potential skills.
        """
        # Step 1: Extract skills using the pre-trained model
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=512)
        outputs = self.model(**inputs)
        logits = outputs.logits
        predictions = logits.argmax(dim=2)
        
        id2label = self.model.config.id2label
        predicted_labels = [id2label[idx.item()] for idx in predictions[0]]
        
        model_skills = []
        i = 0
        while i < len(predicted_labels):
            if predicted_labels[i] == "B-SKILL":
                start = i
                i += 1
                while i < len(predicted_labels) and predicted_labels[i] == "I-SKILL":
                    i += 1
                end = i
                skill_ids = inputs["input_ids"][0][start:end].tolist()
                skill = self.tokenizer.decode(skill_ids, skip_special_tokens=True)
                model_skills.append(skill.strip().lower())
            else:
                i += 1
        
        # Step 2: Extract soft skills using keyword matching
        text_lower = text.lower()
        soft_skills_extracted = set()
        for skill in self.soft_skills:
            if re.search(rf"\b{re.escape(skill)}\b", text_lower):
                soft_skills_extracted.add(skill)
        
        # Step 3: Combine skills
        all_skills = set(model_skills) | soft_skills_extracted
        
        # Step 4: Handle known and potential skills
        if self.skill_set:
            known_skills = [skill for skill in all_skills if skill.lower() in self.skill_set_flat]
            potential_skills = [skill for skill in all_skills if skill.lower() not in self.skill_set_flat]
        else:
            known_skills = list(all_skills)
            potential_skills = list(all_skills)
        
        # Step 5: Prepare output
        if return_grouped and self.skill_set:
            grouped = {}
            for skill in known_skills:
                for cat, cat_skills in self.skill_set.items():
                    if skill.lower() in [s.lower() for s in cat_skills]:
                        if cat not in grouped:
                            grouped[cat] = []
                        grouped[cat].append(skill)
                        break
            known_skills_output = {cat: sorted(skills) for cat, skills in grouped.items()}
        else:
            known_skills_output = sorted(known_skills)
        
        if extract_potential:
            return {
                "known_skills": known_skills_output,
                "potential_skills": sorted(potential_skills)
            }
        return known_skills_output

# Example usage
if __name__ == "__main__":
    # Sample skills.json (optional)
    sample_skills = {
        "Programming Languages": ["Python", "Java", "JavaScript"],
        "Web Development": ["Django", "React"],
        "Soft Skills": ["Communication", "Problem Solving"]
    }
    with open("skills.json", "w") as f:
        json.dump(sample_skills, f)
    
    extractor = JobSkillExtractor(skill_file="skills.json")
    jd_text = """
    About the job
Introduction

In this role, you'll work in one of our IBM Consulting Client Innovation Centers (Delivery Centers), where we deliver deep technical and industry expertise to a wide range of public and private sector clients around the world. Our delivery centers offer our clients locally based skills and technical expertise to drive innovation and adoption of new technology.

A career in IBM Consulting is rooted by long-term relationships and close collaboration with clients across the globe.

You'll work with visionaries across multiple industries to improve the hybrid cloud and AI journey for the most innovative and valuable companies in the world. Your ability to accelerate impact and make meaningful change for your clients is enabled by our strategic partner ecosystem and our robust technology platforms across the IBM portfolio; including Software and Red Hat.

Curiosity and a constant quest for knowledge serve as the foundation to success in IBM Consulting. In your role, you'll be encouraged to challenge the norm, investigate ideas outside of your role, and come up with creative solutions resulting in ground breaking impact for a wide network of clients. Our culture of evolution and empathy centers on long-term career growth and development opportunities in an environment that embraces your unique skills and experience.

Your Role And Responsibilities

We are seeking an experienced Data Scientist. The ideal candidate will have a deep understanding of Machine Learning, Generative AI, and Large Language Models, combined with strong leadership abilities. This role will be responsible for business development, overseeing the development and implementation of advanced data-driven solutions as well as mentor a team of talented data scientists.

Technical Oversight

Oversee the design, development, and deployment of Machine Learning models and AI solutions.
Ensure the use of best practices in data science, including model validation, performance monitoring, and continuous improvement.
Stay informed about the latest developments in AI/ML and drive the adoption of relevant technologies within the team.

Team Development

Provide mentorship and professional development opportunities to project and practice team members.
Actively support in recruitment efforts to attract top talent and grow the team as needed.
Help Foster a culture of learning and continuous development within the AI/Data Science team.

Preferred Education

Master's Degree

Experience

Required technical and professional expertise

7+ years of experience in Data Science and Machine Learning with at least 1 year of experience in conversational solutions design and development using Watson Assistant or similar conversational AI platforms.
Proven track record of delivering AI/ML projects.
Experience with Generative AI and Large Language Models is highly desirable

Technical Skills

Proficiency in programming languages such as Python, R, or Scala.
Deep understanding of Machine Learning frameworks (e.g., TensorFlow, PyTorch) and data processing tools (e.g., SQL, Pandas).
Experience with cloud platforms (e.g., AWS, Azure, Google Cloud) and machine learning deployment.
Strong knowledge of Natural Language Processing (NLP) and AI ethics.

Leadership Skills

Leadership skills with the ability to inspire and motivate junior team members.
Strategic thinking with the ability to align data science initiatives with business objectives.
Very good communication skills, capable of conveying complex ideas to a variety of audiences.
Proven ability to work collaboratively in projects with cross-functional teams.
Experience in building/scaling data science teams and implementing AI solutions in a large (preferably, consulting) organization.
    """
    print("Flat list:", extractor.extract(jd_text))
    print("Grouped:", extractor.extract(jd_text, return_grouped=True))
    print("With potential:", extractor.extract(jd_text, extract_potential=True))

Flat list: ['communication']
Grouped: {'Soft Skills': ['communication']}
With potential: {'known_skills': ['communication'], 'potential_skills': ['collaboration', 'empathy', 'leadership', 'strategic thinking']}
